In [1]:
import metview as mv
from datetime import datetime, timedelta
import os
import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count

Setting MARS_HOME to /usr/local/apps/mars/configs/prod


In [2]:

#
# PRECIPITATION TYPE VERIFICATION - SYNOP MATCHING (DATAFRAME VERSION)
# ---------------------------------------------------------------------
#
# Matches ensemble probability forecasts with SYNOP observations
# Stores matched pairs in pandas DataFrames (Parquet format)
# Optimized for verification analysis
#
# Estibaliz Gascon 12/01/2017
# Revised: 02/01/2025 - DataFrame version with Parquet storage
#
########################################

# ===== CONFIGURATION =====
# Define date range
start_date = datetime(2024, 12, 1)  # Start date: 1 December 2024
end_date = datetime(2024, 12, 18)    # End date: 28 February 2025

# Generate list of dates in YYYYMMDD format
fdates = []
current_date = start_date
while current_date <= end_date:
    fdates.append(int(current_date.strftime('%Y%m%d')))
    current_date += timedelta(days=1)

# Extended forecast steps (0-168h in 3h increments, except after 144h)
lsteps = list(range(0, 145, 3))  # 0, 3, 6, ..., 144
lsteps.extend([150, 156, 162, 168])  # Add the remaining steps

# Path configuration
PATH_OROG_in = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_bias/raw/48r1_2023_esuite/"
PATH_MODEL_in_base = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2024_2025_49r1/"
PATH_OUTPUT = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2024_2025_49r1/"
PATH_SYNOP_in = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/SYNOP/2024_2025/"

# Create output directory
if not os.path.exists(PATH_OUTPUT):
    print(f"Creating output directory: {PATH_OUTPUT}")
    os.makedirs(PATH_OUTPUT, exist_ok=True)

# Verification area (Europe)
AREA = [71, -11, 33, 35]  # [North, West, South, East]

# Orography difference threshold (meters)
OROG_THRESHOLD = 200

# SYNOP weather codes for each precipitation type
SYNOP_CODES = {
    'rain': [51, 52, 53, 54, 55, 58, 59, 60, 61, 62, 63, 64, 65, 68, 69, 
             80, 81, 82, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 99],
    'snow': [70, 71, 72, 73, 74, 75, 76, 77, 78, 85, 86, 93, 94, 95, 97],
    'icepellets': [79, 87, 77],
    'freezrain': [66, 67],
    'freezdrizzle': [56, 57],
    'sleet': [68, 69, 83, 84, 87, 88, 89, 90, 93, 94, 95, 97]
}

# Number of parallel processes
N_PROCESSES = max(1, cpu_count() - 2)

print(f"Processing {len(fdates)} dates from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"Forecast steps: {len(lsteps)} steps")
print(f"Using {N_PROCESSES} parallel processes")
print(f"Output format: Parquet (one file per date)")
print(f"Verification area: {AREA}")
print(f"Orography threshold: ±{OROG_THRESHOLD}m")
print(f"{'='*60}\n")

Processing 18 dates from 2024-12-01 to 2024-12-18
Forecast steps: 53 steps
Using 254 parallel processes
Output format: Parquet (one file per date)
Verification area: [71, -11, 33, 35]
Orography threshold: ±200m



In [2]:
def classify_synop_type(obs_code, precip_type):
    """
    Check if SYNOP observation matches a precipitation type
    
    Parameters:
    -----------
    obs_code : float or array
        SYNOP present weather code
    precip_type : str
        Type to check ('rain', 'snow', etc.)
    
    Returns:
    --------
    bool or array : True if observation matches type
    """
    result = obs_code == SYNOP_CODES[precip_type][0]
    for code in SYNOP_CODES[precip_type][1:]:
        result = result | (obs_code == code)
    return result


def process_single_date(number_date):
    """
    Process a single date and return DataFrame with all matched pairs
    
    Parameters:
    -----------
    number_date : int
        Date in YYYYMMDD format
    
    Returns:
    --------
    tuple : (date, n_records, success)
    """
    fdate = number_date
    PATH_MODEL_in = PATH_MODEL_in_base + str(fdate) + "_0/"
    
    # Check if model input directory exists
    if not os.path.exists(PATH_MODEL_in):
        return (fdate, 0, False)
    
    # List to store data for all steps
    all_data = []
    
    # Process each forecast step
    for step in lsteps:
        
        try:
            # Read probability files for all precipitation types
            rain_filter = mv.read(PATH_MODEL_in + f"perc_rain_{step}.grib")
            snow_filter = mv.read(PATH_MODEL_in + f"perc_snow_{step}.grib")
            wetsnow_filter = mv.read(PATH_MODEL_in + f"perc_wetsnow_{step}.grib")
            freezrain_filter = mv.read(PATH_MODEL_in + f"perc_freezrain_{step}.grib")
            freezdrizzle_filter = mv.read(PATH_MODEL_in + f"perc_freezdrizzle_{step}.grib")
            icepellets_filter = mv.read(PATH_MODEL_in + f"perc_icepellets_{step}.grib")
            sleet_filter = mv.read(PATH_MODEL_in + f"perc_sleet_{step}.grib")
            
            # Read orography
            orog = mv.read(PATH_OROG_in + "orog.grib")
            
        except Exception as e:
            continue
        
        # Get valid date from the forecast field
        fdate_SYN = int(mv.yyyymmdd(mv.valid_date(rain_filter)))
        valid_time = mv.valid_date(rain_filter)
        
        # Calculate SYNOP time (hour of day, 0-23)
        lstep = step % 24
        
        # Read SYNOP observations
        try:
            SYNOP_data = mv.read(PATH_SYNOP_in + f"SYNOP_mannual_{fdate_SYN}.bufr")
        except Exception as e:
            continue
        
        # Filter SYNOP observations
        obs_max = mv.obsfilter(
            output="geopoints",
            parameter=20003,  # Present weather
            time=lstep,
            location_filter="area",
            area=AREA,
            data=SYNOP_data
        )
        
        observ_height = mv.obsfilter(
            output="geopoints",
            parameter=7001,  # Height of station
            time=lstep,
            location_filter="area",
            area=AREA,
            data=SYNOP_data
        )
        
        # Quality control: Filter by orography difference
        # First, ensure obs_height has the same points as obs_max
        obs_height = mv.subsample(observ_height, obs_max)
        
        # Check if we have matching sizes
        try:
            n_weather = len(mv.values(obs_max))
            n_height = len(mv.values(obs_height))
        except:
            continue
        
        # Skip if sizes don't match (shouldn't happen with subsample, but be safe)
        if n_weather != n_height:
            continue
        
        Locations_orog = mv.nearest_gridpoint(orog, obs_max)
        
        # Calculate orography difference
        diff_orog = obs_height - Locations_orog
        
        # Create filter for orography threshold
        diff_orog_ok = (diff_orog < OROG_THRESHOLD) & (diff_orog > -OROG_THRESHOLD)
        
        # Apply orography filter
        obs_ = mv.filter(obs_max, diff_orog_ok == 1)
        obs_height_ = mv.filter(obs_height, diff_orog_ok == 1)
        
        # Remove missing or invalid observations
        Miss = (obs_ == 508) | (obs_ == 509) | (obs_ == 510) | (obs_ == 511) | (obs_ > 100)
        obs = mv.filter(obs_, Miss == 0)
        obs_height_filtered = mv.filter(obs_height_, Miss == 0)
        
        # Get observation data
        try:
            obs_values = mv.values(obs)
            obs_lats = mv.latitudes(obs)
            obs_lons = mv.longitudes(obs)
            obs_elevs = mv.values(obs_height_filtered)
            n_obs = len(obs_values)
        except:
            continue
        
        if n_obs == 0:
            continue
        
        # Extract model probability values at observation locations
        prob_rain = mv.values(mv.nearest_gridpoint(rain_filter, obs))
        prob_snow = mv.values(mv.nearest_gridpoint(snow_filter, obs))
        prob_wetsnow = mv.values(mv.nearest_gridpoint(wetsnow_filter, obs))
        prob_sleet = mv.values(mv.nearest_gridpoint(sleet_filter, obs))
        prob_freezrain = mv.values(mv.nearest_gridpoint(freezrain_filter, obs))
        prob_freezdrizzle = mv.values(mv.nearest_gridpoint(freezdrizzle_filter, obs))
        prob_icepellets = mv.values(mv.nearest_gridpoint(icepellets_filter, obs))
        
        # Classify observations
        obs_rain = np.array([classify_synop_type(obs_values[i], 'rain') for i in range(n_obs)], dtype=int)
        obs_snow = np.array([classify_synop_type(obs_values[i], 'snow') for i in range(n_obs)], dtype=int)
        obs_icepellets = np.array([classify_synop_type(obs_values[i], 'icepellets') for i in range(n_obs)], dtype=int)
        obs_freezrain = np.array([classify_synop_type(obs_values[i], 'freezrain') for i in range(n_obs)], dtype=int)
        obs_freezdrizzle = np.array([classify_synop_type(obs_values[i], 'freezdrizzle') for i in range(n_obs)], dtype=int)
        obs_sleet = np.array([classify_synop_type(obs_values[i], 'sleet') for i in range(n_obs)], dtype=int)
        
        # Create DataFrame for this step
        step_data = pd.DataFrame({
            # Metadata
            'forecast_date': fdate,
            'valid_date': fdate_SYN,
            'forecast_step': step,
            'valid_hour': lstep,
            
            # Station information
            'latitude': obs_lats,
            'longitude': obs_lons,
            'elevation': obs_elevs,
            
            # SYNOP observation
            'synop_code': obs_values,
            
            # Observed precipitation type (binary flags)
            'obs_rain': obs_rain,
            'obs_snow': obs_snow,
            'obs_wetsnow': 0,  # Not separately classified in SYNOP
            'obs_sleet': obs_sleet,
            'obs_freezrain': obs_freezrain,
            'obs_freezdrizzle': obs_freezdrizzle,
            'obs_icepellets': obs_icepellets,
            
            # Model probabilities (%)
            'prob_rain': prob_rain,
            'prob_snow': prob_snow,
            'prob_wetsnow': prob_wetsnow,
            'prob_sleet': prob_sleet,
            'prob_freezrain': prob_freezrain,
            'prob_freezdrizzle': prob_freezdrizzle,
            'prob_icepellets': prob_icepellets,
        })
        
        all_data.append(step_data)
    
    # Combine all steps for this date
    if len(all_data) > 0:
        date_df = pd.concat(all_data, ignore_index=True)
        
        # Save to Parquet file (compressed, efficient)
        output_file = PATH_OUTPUT + f"matched_pairs_{fdate}.parquet"
        date_df.to_parquet(output_file, compression='snappy', index=False)
        
        return (fdate, len(date_df), True)
    else:
        return (fdate, 0, False)


# ===== MAIN PARALLEL PROCESSING =====
if __name__ == '__main__':
    
    print(f"Starting parallel processing...")
    start_time = datetime.now()
    
    # Create a pool of worker processes
    with Pool(processes=N_PROCESSES) as pool:
        results = pool.map(process_single_date, fdates)
    
    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    
    # Summarize results
    print(f"\n{'='*60}")
    print("PROCESSING COMPLETE!")
    print(f"{'='*60}")
    print(f"Total time: {elapsed/60:.1f} minutes ({elapsed:.0f} seconds)")
    print(f"\nResults by date:")
    print(f"{'Date':<12} {'Records':<12} {'Status':<12}")
    print(f"{'-'*60}")
    
    total_records = 0
    successful_dates = 0
    
    for fdate, n_records, success in results:
        total_records += n_records
        if success:
            successful_dates += 1
        status = "✓ Success" if success else "✗ Failed"
        print(f"{fdate:<12} {n_records:<12} {status:<12}")
    
    print(f"{'-'*60}")
    print(f"{'TOTAL':<12} {total_records:<12} {successful_dates}/{len(fdates)} dates")
    print(f"\nTotal matched pairs: {total_records:,}")
    print(f"Successful dates: {successful_dates}/{len(fdates)} ({100*successful_dates/len(fdates):.1f}%)")
    print(f"Average records per date: {total_records/len(fdates):.0f}")
    print(f"Average time per date: {elapsed/len(fdates):.1f} seconds")
    
    # Summary of output
    print(f"\n{'='*60}")
    print("OUTPUT FILES")
    print(f"{'='*60}")
    print(f"Location: {PATH_OUTPUT}")
    print(f"Format: Parquet (compressed)")
    print(f"Files created: {successful_dates} files (one per date)")
    print(f"File naming: matched_pairs_YYYYMMDD.parquet")
    print(f"\nEach file contains:")
    print(f"  - Station coordinates (lat, lon, elevation)")
    print(f"  - Forecast metadata (date, step, valid time)")
    print(f"  - SYNOP observations (weather codes + binary flags)")
    print(f"  - Model probabilities for all precipitation types")
    print(f"  - All matched pairs for that date (all steps)")


Starting parallel processing...


Process ForkPoolWorker-253:
Process ForkPoolWorker-251:
Process ForkPoolWorker-19:
Process ForkPoolWorker-218:
Process ForkPoolWorker-197:
Process ForkPoolWorker-220:
Process ForkPoolWorker-235:
Process ForkPoolWorker-196:
Process ForkPoolWorker-190:
Process ForkPoolWorker-181:
Process ForkPoolWorker-36:
Process ForkPoolWorker-234:
Process ForkPoolWorker-54:
Process ForkPoolWorker-198:
Process ForkPoolWorker-55:
Process ForkPoolWorker-242:
Process ForkPoolWorker-40:
Process ForkPoolWorker-219:
Process ForkPoolWorker-8:
Process ForkPoolWorker-24:
Process ForkPoolWorker-53:
Process ForkPoolWorker-210:
Process ForkPoolWorker-119:
Process ForkPoolWorker-42:
Process ForkPoolWorker-212:
Process ForkPoolWorker-41:
Process ForkPoolWorker-175:
Process ForkPoolWorker-97:
Process ForkPoolWorker-232:
Process ForkPoolWorker-33:
Process ForkPoolWorker-240:
Process ForkPoolWorker-217:
Process ForkPoolWorker-189:
Process ForkPoolWorker-238:
Process ForkPoolWorker-160:
Process ForkPoolWorker-86:
Proces

PythonServe - FATAL  - 20260102.132319 - Signal 2 received
Python - FATAL  - 20260102.132319 - Server localhost port 53493 is dead



KeyboardInterrupt


KeyboardInterrupt

